Validating a prediction model, like `cardiffnlp/twitter-xlm-roberta-base-sentiment`, requires choosing an adequate sample size to ensure that the performance estimates (like accuracy or error rate) have tight and precise confidence intervals *(Riley et al., 2024)*. In standard statistics, reviewing just 384 comments is enough to represent the entire 203,994-row dataset with a 95% confidence level and a 5% margin of error.

# Setup

In [1]:
import pandas as pd

from src.config import INTERIM_DATA_DIR

In [2]:
df_sentiment_result = pd.read_parquet(INTERIM_DATA_DIR / "df_sentiment_result.parquet")

df_sentiment_result.sample(10)

,video_id,comment_id,text,language,sentiment_label
12495,Smgql_DESn8,UgwbP4YCwPvzCIrKRFt4AaABAg,"제가 제일 좋아하는 노래예요, 영원히 사랑할 거예요! 🤍🤍😭🤍😭😭🫶🏻🫶🏻🫶🏻",kor_Hang,positive
56187,uCKNo3KitNw,Ugx44EoxMipjnZb9_WV4AaABAg,Chan is like abby in saja boys😭,eng_Latn,neutral
93150,gJRVycxNtVI,UgxUF2xqk6389ip7ktp4AaABAg,YEP!!!! OH MY GAH HOW DID LEE KNOW BECOME SO R...,yue_Hant,positive
123656,jld2n72GLIw,Ugyhnm-charA56XFXTh4AaABAg,미래를 보고 온걸까,kor_Hang,neutral
146627,PVQv8w50Uxo,UgyVY7KqXeyyXRyb45F4AaABAg,20:31 Chan avoiding Lee Know's angry eyes 🤣🤣🤣🤣,eng_Latn,neutral
200671,gSffzttJnc4,UgzXXg6tyTlDJ7vDbfl4AaABAg,I love stray kids ❤❤❤❤❤❤❤,eng_Latn,positive
8091,y-rPpGpnSg0,Ugw7Zd0cFf7wOr_Mrcd4AaABAg,I came from X (Twitter) because they drag this...,eng_Latn,negative
83135,lldrm_7fVP0,UgxnzWyqqtVeAgq-Y-Z4AaABAg,Stray KidsのLIVEに１度でも行ったら、抜け出せない🥹✨\n私はHyunjinペン...,jpn_Jpan,positive
185622,FnWKcU7ayH0,UgzOIvpDwkarXw3z4nd4AaABAg,"l don’t understand kerea at all,but honestly,t...",eng_Latn,positive
60320,y19Ae0CDZU4,Ugx9gufqTwuIBTWgMV14AaABAg,This man is just hilarious his vlogs are soo f...,eng_Latn,positive


# Evaluation 1

In [ ]:
# Extract 385 random rows as samples
sample_df_sentiment_result = df_sentiment_result.drop("sentiment_label", axis=1).sample(n=384, random_state=8)

sample_df_sentiment_result.head()

,video_id,comment_id,text,language
135722,yY8PmXgmbak,Ugyp56EN4Z8gixiZ2Et4AaABAg,Felix is so cute with his deep voice 💗😭,eng_Latn
181455,UAanagaR_jY,UgzlvHzIxxc7fY4Bw_94AaABAg,SEUNGMINS SO CUTE BEING THE ONLY ONE SITTING D...,yue_Hant
13100,Wwdcv8TRbcA,UgwC0hsu1YekAYIDlrx4AaABAg,SEUNGMIN IS BEST VOCALIST\nSEUNGMIN IS BEST DA...,yue_Hant
79479,k_Er4BnLCFY,UgxLpXq6Z_LuOmRFRKN4AaABAg,LEEKNOW AT THE END WHEN HIS FACE WAS SAUR NEAR...,yue_Hant
110803,ADOxqrfKfyE,Ugy8ywQUTTCaO6MyitR4AaABAg,*”step out of them voices”*,eng_Latn


In [ ]:
# Save the sample data as an Excel file to manually label the true sentiment results
sample_df_sentiment_result.to_excel(INTERIM_DATA_DIR / "sample_df_sentiment_result.xlsx")

In [ ]:
# Load the manually labeled sample data
df_true_sentiment_result = pd.read_excel(INTERIM_DATA_DIR / "df_true_sentiment_result.xlsx")

In [34]:
df_true_sentiment_result.head(5)

,Unnamed: 0,video_id,comment_id,text,language,true_sentiment_scores,true_sentiment_label
0,135722,yY8PmXgmbak,Ugyp56EN4Z8gixiZ2Et4AaABAg,Felix is so cute with his deep voice 💗😭,eng_Latn,2,positive
1,181455,UAanagaR_jY,UgzlvHzIxxc7fY4Bw_94AaABAg,SEUNGMINS SO CUTE BEING THE ONLY ONE SITTING D...,yue_Hant,2,positive
2,13100,Wwdcv8TRbcA,UgwC0hsu1YekAYIDlrx4AaABAg,SEUNGMIN IS BEST VOCALIST\nSEUNGMIN IS BEST DA...,yue_Hant,2,positive
3,79479,k_Er4BnLCFY,UgxLpXq6Z_LuOmRFRKN4AaABAg,LEEKNOW AT THE END WHEN HIS FACE WAS SAUR NEAR...,yue_Hant,2,positive
4,110803,ADOxqrfKfyE,Ugy8ywQUTTCaO6MyitR4AaABAg,*”step out of them voices”*,eng_Latn,1,neutral


In [36]:
labeled_df = df_true_sentiment_result.join(df_sentiment_result[["sentiment_label"]], how="left")

labeled_df.head()

,Unnamed: 0,video_id,comment_id,text,language,true_sentiment_scores,true_sentiment_label,sentiment_label
0,135722,yY8PmXgmbak,Ugyp56EN4Z8gixiZ2Et4AaABAg,Felix is so cute with his deep voice 💗😭,eng_Latn,2,positive,positive
1,181455,UAanagaR_jY,UgzlvHzIxxc7fY4Bw_94AaABAg,SEUNGMINS SO CUTE BEING THE ONLY ONE SITTING D...,yue_Hant,2,positive,positive
2,13100,Wwdcv8TRbcA,UgwC0hsu1YekAYIDlrx4AaABAg,SEUNGMIN IS BEST VOCALIST\nSEUNGMIN IS BEST DA...,yue_Hant,2,positive,neutral
3,79479,k_Er4BnLCFY,UgxLpXq6Z_LuOmRFRKN4AaABAg,LEEKNOW AT THE END WHEN HIS FACE WAS SAUR NEAR...,yue_Hant,2,positive,negative
4,110803,ADOxqrfKfyE,Ugy8ywQUTTCaO6MyitR4AaABAg,*”step out of them voices”*,eng_Latn,1,neutral,positive


In [37]:
# Identify the misclassified rows
misclassified = labeled_df[labeled_df["sentiment_label"] != labeled_df["true_sentiment_label"]]

# Calculate the error rate
error_rate = len(misclassified) / len(labeled_df)
print(f"Estimated Error Rate: {error_rate * 100:.2f}%")

Estimated Error Rate: 41.15%


***Notes:*** 
* A 41.15% error rate is brutal. At this point, the model is only performing slightly better than a coin flip, which makes it useless for a serious data analysis project.
* There are not even true negative comments in the sample data. This might be because most of the viewers are the true fans who often drop a lot of complements or praise. 
* From these findings, we can conclude that we cannot really find insights from negative comments as most of the viewers are positive about the video.
* The model is also not performing well, so we may need to consider other models or fine-tuning the current model to improve its performance.

Problem with YouTube Engagement Rate Calculation for Music Videos:
* Music videos get massive view counts but most viewers are listening, not engaging. People play a song on repeat; they don't like it 5 times. High views, low interaction.
[Source](https://sociavault.com/blog/good-engagement-rate-youtube)

# Creating A Sample Dataset for Fine-Tuning

## Positives - 1

In [15]:
# Get 300 positive-labeled comments only 
sample_df_positive_sentiment_result_1 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "positive"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=1)
)

sample_df_positive_sentiment_result_1.head()

,video_id,comment_id,text,language
50586,zBvKNlUO0b4,UgwZj9XiXtcw8ZLFgQJ4AaABAg,"Chan: *thinks all his choices throughly, and r...",eng_Latn
32845,m_M5Uo39xE0,UgwOg9louI6aur5u9QV4AaABAg,세상에 😭😭 이건 정말 달콤해 내 인생에서 본 최고의 뮤직비디오 중 하나야 😭❤,kor_Hang
165842,LIjxH_pY2Ew,UgzbyRFRCPSOgfTZFwt4AaABAg,This trio has the best chemistry!,eng_Latn
113926,iT_E9IgAQFM,UgyBJNbGOWLOyRUB9W14AaABAg,"The amount of times I had to repeat, pause, pl...",eng_Latn
131793,8Dno0lrIfJE,UgymQRsxVCnf7iv54494AaABAg,Yes!!! The Kids left no crumbs with these shows!,eng_Latn


In [16]:
# Get the comments only and save them
positive_labeled_comments_1 = sample_df_positive_sentiment_result_1[["text"]]

positive_labeled_comments_1.to_json(INTERIM_DATA_DIR / "positive_labeled_comments_1.json")

In [11]:
# Get the result of manually labeled comments
true_positive_labeled_comments_1 = pd.read_json(INTERIM_DATA_DIR / "true_positive_labeled_comments_1.json")
true_positive_labeled_comments_1.index = true_positive_labeled_comments_1.index.astype(int)
print(true_positive_labeled_comments_1["label"].value_counts())
true_positive_labeled_comments_1.head()

label
2    281
1     19
Name: count, dtype: int64


,label
50586,2
32845,2
165842,2
113926,2
131793,2


In [18]:
# Select only the result of positive-manually labeled comments
selected_true_positive_labeled_comments_1 = (
    true_positive_labeled_comments_1[true_positive_labeled_comments_1["label"] == 2]
)

## Positives - 2

In [ ]:
# Get other 300 positive-labeled comments randomly
sample_df_positive_sentiment_result_2 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "positive"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=2)
)

sample_df_positive_sentiment_result_2.head()

,video_id,comment_id,text,language
140649,uJRNtpcsnUI,UgyScaib524HwBVFng94AaABAg,I have never seen Chan hold back on his dancin...,eng_Latn
10953,U35ePIHz85Y,UgwAQyv1nAaPDVrO5fh4AaABAg,Jeongin and Minho looks so good together.\n\nI...,eng_Latn
79188,MPI5pikwcMI,UgxLKr6z3y0eJePVssN4AaABAg,그는 정말 잘 움직여요 🤧💗💗🩷🩷🩷,kor_Hang
181948,QUzIW08qNPg,Ugzm9EXW-fsw18yawe94AaABAg,"Happy lunar new year to Straykids and STAY, I ...",eng_Latn
140376,RQG3DBFuju0,UgyS3H62ybwZSI_p7_B4AaABAg,12:18 HELLOOO??? how is he literally so pretty🥹,eng_Latn


In [20]:
# Get the comments only and save them
positive_labeled_comments_2 = sample_df_positive_sentiment_result_2[["text"]]

positive_labeled_comments_2.to_json(INTERIM_DATA_DIR / "positive_labeled_comments_2.json")

In [ ]:
# Get the result of manually labeled comments
true_positive_labeled_comments_2 = pd.read_json(INTERIM_DATA_DIR / "true_positive_labeled_comments_2.json")
true_positive_labeled_comments_2.index = true_positive_labeled_comments_2.index.astype(int)
print(true_positive_labeled_comments_2["label"].value_counts())
true_positive_labeled_comments_2.head()

label
2    296
1      4
Name: count, dtype: int64


,label
140649,2
10953,2
79188,2
181948,2
140376,2


In [26]:
# Select only the result of positive-manually labeled comments
selected_true_positive_labeled_comments_2 = (
    true_positive_labeled_comments_2[true_positive_labeled_comments_2["label"] == 2]
)

## Combining the Positives

In [ ]:
selected_true_positive_labeled_comments = pd.concat([selected_true_positive_labeled_comments_1, selected_true_positive_labeled_comments_2])

selected_true_positive_labeled_comments = (
    selected_true_positive_labeled_comments
    .reset_index().drop_duplicates().set_index("index").sample(300)
)

selected_true_positive_labeled_comments.index.name = None

print(selected_true_positive_labeled_comments.shape)
print(selected_true_positive_labeled_comments["label"].value_counts())
selected_true_positive_labeled_comments.head()

(300, 1)


,label
129312,1
81260,2
125334,2
124618,2
14789,2


## Neutrals - 1

In [10]:
# Get 300 neutral-labeled comments only 
sample_df_neutral_sentiment_result_1 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "neutral"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=1)
)

print(sample_df_neutral_sentiment_result_1.shape)
sample_df_neutral_sentiment_result_1.head()

(300, 4)


,video_id,comment_id,text,language
172598,JVM65SjoZSw,UgzgBLC7KPb9D0SkteR4AaABAg,ESCAPE is literally RED LIGHTS twin,eng_Latn
193169,4rs98PrwXX0,UgztAPylikJwesIcE-p4AaABAg,STRAY KIDS EVERYWHERE ALL AROUND THE WORLD 🗣️🔥,yue_Hant
200943,LIjxH_pY2Ew,Ugzy4t2E5AAOSDwLVRJ4AaABAg,6:45 Seungmin chasing the birds again,eng_Latn
45642,9EVsOUqpQB8,UgwWfGa9OK50vq2LEfZ4AaABAg,15:00 Seeing han sleeping with his jacket look...,eng_Latn
30169,cq5CHXCvmGM,UgwMPF972wAfKd2zLmt4AaABAg,jeongin: *picks a very nice amusement park for...,eng_Latn


In [11]:
# Get the comments only and save them
neutral_labeled_comments_1 = sample_df_neutral_sentiment_result_1[["text"]]

neutral_labeled_comments_1.to_json(INTERIM_DATA_DIR / "neutral_labeled_comments_1.json")

In [13]:
# Get the result of manually labeled comments
true_neutral_labeled_comments_1 = pd.read_json(INTERIM_DATA_DIR / "true_neutral_labeled_comments_1.json")
true_neutral_labeled_comments_1.index = true_neutral_labeled_comments_1.index.astype(int)
print(true_neutral_labeled_comments_1["label"].value_counts())
true_neutral_labeled_comments_1.head()

label
2    156
1    139
0      4
Name: count, dtype: int64


,label
172598,2
193169,2
200943,1
45642,2
30169,1


In [13]:
# Select only the result of neutral-manually labeled comments
selected_true_neutral_labeled_comments_1 = (
    true_neutral_labeled_comments_1[true_neutral_labeled_comments_1["label"] == 1]
)

## Neutrals - 2

In [3]:
# Get 300 neutral-labeled comments only 
sample_df_neutral_sentiment_result_2 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "neutral"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=2)
)

print(sample_df_neutral_sentiment_result_2.shape)
sample_df_neutral_sentiment_result_2.head()

(300, 4)


,video_id,comment_id,text,language
6051,WKVcSrMiuOU,Ugw5IuMlynY4XSB42Xh4AaABAg,Quien volvió a este video por 2KidsRoom para v...,spa_Latn
27985,6PmBVzTAEUk,UgwleZ7ieupQZas6iq94AaABAg,13:31 CHANNIE,kor_Hang
4262,eOrl2slx0AQ,Ugw3CoXsV1m1wfweL2d4AaABAg,2:54 🐶🐖🐇応援に来ました,jpn_Jpan
106717,Ib2jHyVChxw,Ugy3srO-390-4EqRKFV4AaABAg,🐶🐖🐇絡み\n9:58 ★10:13 11:08 11:46 13:30 13:56 15:...,eng_Latn
129395,GNfMfTZWJ-Y,UgylBsMB1Yd3o-zMeeJ4AaABAg,펠릭스 아임 어 스테이,kor_Hang


In [4]:
# Get the comments only and save them
neutral_labeled_comments_2 = sample_df_neutral_sentiment_result_2[["text"]]

neutral_labeled_comments_2.to_json(INTERIM_DATA_DIR / "neutral_labeled_comments_2.json")

In [14]:
# Get the result of manually labeled comments
true_neutral_labeled_comments_2 = pd.read_json(INTERIM_DATA_DIR / "true_neutral_labeled_comments_2.json")
true_neutral_labeled_comments_2.index = true_neutral_labeled_comments_2.index.astype(int)
print(true_neutral_labeled_comments_2["label"].value_counts())
true_neutral_labeled_comments_2.head()

label
2    240
1     55
0      5
Name: count, dtype: int64


,label
6051,1
27985,2
4262,2
106717,1
129395,2


In [6]:
# Select only the result of neutral-manually labeled comments
selected_true_neutral_labeled_comments_2 = (
    true_neutral_labeled_comments_2[true_neutral_labeled_comments_2["label"] == 1]
)

## Neutrals - 3

In [7]:
# Get 300 neutral-labeled comments only 
sample_df_neutral_sentiment_result_3 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "neutral"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=3)
)

print(sample_df_neutral_sentiment_result_3.shape)
sample_df_neutral_sentiment_result_3.head()

(300, 4)


,video_id,comment_id,text,language
100094,T72rVLtXW2A,UgxYpRPJ1nfx9M2z1lJ4AaABAg,Y’all I heard Duolingo 7:31,eng_Latn
114959,YsvbN665LZE,Ugyc5G0anpzafNUkgDl4AaABAg,How many international fans are supporting HAN...,eng_Latn
119310,JSjJCYFr6ic,UgyEWUyIPwblcbi59KV4AaABAg,One day I will go to a Stray Kids concert,eng_Latn
144545,w_XIdjXgsjE,UgyuOdpI9RqNEoo843F4AaABAg,YONGBOK OUTFIT>>>>>>>>>>>,yue_Hant
37277,NyZ0iYlTvRk,UgwR6Bq4wrz6dDw0-OV4AaABAg,no one:\nnot even chan's laptop:\nChanLix: *Ki...,eng_Latn


In [8]:
# Get the comments only and save them
neutral_labeled_comments_3 = sample_df_neutral_sentiment_result_3[["text"]]

neutral_labeled_comments_3.to_json(INTERIM_DATA_DIR / "neutral_labeled_comments_3.json")

In [15]:
# Get the result of manually labeled comments
true_neutral_labeled_comments_3 = pd.read_json(INTERIM_DATA_DIR / "true_neutral_labeled_comments_3.json")
true_neutral_labeled_comments_3.index = true_neutral_labeled_comments_3.index.astype(int)
print(true_neutral_labeled_comments_3["label"].value_counts())
true_neutral_labeled_comments_3.head()

label
1    171
2    124
0      5
Name: count, dtype: int64


,label
100094,1
114959,2
119310,2
144545,2
37277,1


In [14]:
# Select only the result of neutral-manually labeled comments
selected_true_neutral_labeled_comments_3 = (
    true_neutral_labeled_comments_3[true_neutral_labeled_comments_3["label"] == 1]
)

## Combining the Neutrals

In [17]:
selected_true_neutral_labeled_comments = pd.concat([selected_true_neutral_labeled_comments_1, selected_true_neutral_labeled_comments_2, selected_true_neutral_labeled_comments_3])

selected_true_neutral_labeled_comments = (
    selected_true_neutral_labeled_comments
    .reset_index().drop_duplicates().set_index("index").sample(300)
)

selected_true_neutral_labeled_comments.index.name = None

print(selected_true_neutral_labeled_comments.shape)
print(selected_true_neutral_labeled_comments["label"].value_counts())
selected_true_neutral_labeled_comments.head()

(300, 1)
label
1    300
Name: count, dtype: int64


,label
194088,1
72132,1
44298,1
171125,1
74650,1


## Negatives - 1

In [3]:
# Get 300 negative-labeled comments only 
sample_df_negative_sentiment_result_1 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "negative"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=1)
)

print(sample_df_negative_sentiment_result_1.shape)
sample_df_negative_sentiment_result_1.head()

(300, 4)


,video_id,comment_id,text,language
111471,o66KJX1YJ3s,Ugy9w_2rLLs8jkYMV854AaABAg,12:18 - I.N saying he looks ugly while looking...,eng_Latn
34782,yQqQnpyfIUY,Ugwpm1xI2bcFqO4yeH14AaABAg,Stays: **crying over Hyunjin looking so freaki...,eng_Latn
14419,wgfPkWSDUHU,UgwCuvjTCoI3XrhDysN4AaABAg,Non stays will never understand how much this ...,eng_Latn
31183,MgFlQ9Hquz4,UgwndVvEnDaksKQMbSV4AaABAg,When Hyunjin and Seungmin gather together is d...,eng_Latn
90076,50WosZs46mg,UgxSGC8J621mdexuUdN4AaABAg,NOT THEM JUST SPAWNING OUT OF THE WATER 😭😭,yue_Hant


In [4]:
# Get the comments only and save them
negative_labeled_comments_1 = sample_df_negative_sentiment_result_1[["text"]]

negative_labeled_comments_1.to_json(INTERIM_DATA_DIR / "negative_labeled_comments_1.json")

In [16]:
# Get the result of manually labeled comments
true_negative_labeled_comments_1 = pd.read_json(INTERIM_DATA_DIR / "true_negative_labeled_comments_1.json")
true_negative_labeled_comments_1.index = true_negative_labeled_comments_1.index.astype(int)
print(true_negative_labeled_comments_1["label"].value_counts())
true_negative_labeled_comments_1.head()

label
2    154
1     96
0     50
Name: count, dtype: int64


,label
111471,2
34782,2
14419,2
31183,2
90076,2


In [6]:
# Select only the result of neutral-manually labeled comments
selected_true_negative_labeled_comments_1 = (
    true_negative_labeled_comments_1[true_negative_labeled_comments_1["label"] == 0]
)

## Negatives - 2

In [7]:
# Get 300 negative-labeled comments only 
sample_df_negative_sentiment_result_2 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "negative"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=2)
)

print(sample_df_negative_sentiment_result_2.shape)
sample_df_negative_sentiment_result_2.head()

(300, 4)


,video_id,comment_id,text,language
122032,eCS6BPtxSTg,Ugygn8tlis-FsUSkFMh4AaABAg,"""Lee know work out for his health. Changbin wo...",eng_Latn
143536,tFtCkTEgc4s,UgyU06IbpR_FFuSTMYp4AaABAg,on a totally unrelated note i hope felix can b...,eng_Latn
69289,RCsg2mF0C9E,UgxFcOk7KXYlDP4s7ox4AaABAg,ENGLISH SUBS PLZ???!!!!?????,kor_Hang
128702,gSffzttJnc4,UgyKtdq69aM_HXh6A1V4AaABAg,the skzoos 🫶🫶,eng_Latn
169796,vUulNBcOOeI,UgzEJCno7CH6BlzQpnF4AaABAg,I’m gonna cry. Han is so relatable to me with ...,eng_Latn


In [8]:
# Get the comments only and save them
negative_labeled_comments_2 = sample_df_negative_sentiment_result_2[["text"]]

negative_labeled_comments_2.to_json(INTERIM_DATA_DIR / "negative_labeled_comments_2.json")

In [17]:
# Get the result of manually labeled comments
true_negative_labeled_comments_2 = pd.read_json(INTERIM_DATA_DIR / "true_negative_labeled_comments_2.json")
true_negative_labeled_comments_2.index = true_negative_labeled_comments_2.index.astype(int)
print(true_negative_labeled_comments_2["label"].value_counts())
true_negative_labeled_comments_2.head()

label
2    159
1     86
0     55
Name: count, dtype: int64


,label
122032,1
143536,0
69289,0
128702,2
169796,2


In [10]:
# Select only the result of negative-manually labeled comments
selected_true_negative_labeled_comments_2 = (
    true_negative_labeled_comments_2[true_negative_labeled_comments_2["label"] == 0]
)

## Negatives - 3

In [11]:
# Get 300 negative-labeled comments only 
sample_df_negative_sentiment_result_3 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "negative"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=3)
)

print(sample_df_negative_sentiment_result_3.shape)
sample_df_negative_sentiment_result_3.head()

(300, 4)


,video_id,comment_id,text,language
11556,7KU2yybAO5U,UgwB3ncDeXoOmo-zLid4AaABAg,Why am I early when I don't even understand wh...,eng_Latn
45624,5mncENzOz9w,UgwwEzmmgfglGjrkFY14AaABAg,Guys pls make english subtitles for their live...,eng_Latn
87485,iWlfaLxgIas,UgxqTUY7NAFGkD6awn54AaABAg,17:15 CHANGBINS ENGLISH IS SO ATTRACTIVE OMGGG??,yue_Hant
132906,c6yyoWhyiWw,UgyNGUA7Aw6qLGM5me94AaABAg,"15:04 Chan´s cute giggles in the background :,(",eng_Latn
69886,rpHztlgROH8,Ugxfp9Gf93raGaTfn9N4AaABAg,"People says that. "" stray kids has only humans...",eng_Latn


In [12]:
# Get the comments only and save them
negative_labeled_comments_3 = sample_df_negative_sentiment_result_3[["text"]]

negative_labeled_comments_3.to_json(INTERIM_DATA_DIR / "negative_labeled_comments_3.json")

In [18]:
# Get the result of manually labeled comments
true_negative_labeled_comments_3 = pd.read_json(INTERIM_DATA_DIR / "true_negative_labeled_comments_3.json")
true_negative_labeled_comments_3.index = true_negative_labeled_comments_3.index.astype(int)
print(true_negative_labeled_comments_3["label"].value_counts())
true_negative_labeled_comments_3.head()

label
1    136
2    126
0     37
Name: count, dtype: int64


,label
11556,0
45624,0
87485,2
132906,2
69886,1


In [14]:
# Select only the result of negative-manually labeled comments
selected_true_negative_labeled_comments_3 = (
    true_negative_labeled_comments_3[true_negative_labeled_comments_3["label"] == 0]
)

## Negatives - 4

In [15]:
# Get 300 negative-labeled comments only 
sample_df_negative_sentiment_result_4 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "negative"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=4)
)

print(sample_df_negative_sentiment_result_4.shape)
sample_df_negative_sentiment_result_4.head()

(300, 4)


,video_id,comment_id,text,language
28984,Rj4rhKrQ7x0,UgwlykHRh4uN5FU5LwF4AaABAg,HELP CHANS EXPRESSION SCARED ME😭🙏🏻,yue_Hant
139543,KfUfcBPLteg,UgyRmFx4nt86xuwE1Gl4AaABAg,The “I can’t sleep” part is me right now. It’s...,eng_Latn
18746,8_SnsP5j9aA,UgwfL7Zmrp9kxrqYxjZ4AaABAg,Skz: *staring at the camera\n\nMe: *smiling li...,eng_Latn
118604,De9zCM63beg,UgyEijyrvzZpuQPfqDZ4AaABAg,8:02 gosh hannie was the tiniest bug that day\...,eng_Latn
90156,xcMM_5j1GK8,UgxSI5VQdrWXOUBPuG14AaABAg,Y YO AQUÍ SIN DINERO 😭👊,kor_Hang


In [16]:
# Get the comments only and save them
negative_labeled_comments_4 = sample_df_negative_sentiment_result_4[["text"]]

negative_labeled_comments_4.to_json(INTERIM_DATA_DIR / "negative_labeled_comments_4.json")

In [19]:
# Get the result of manually labeled comments
true_negative_labeled_comments_4 = pd.read_json(INTERIM_DATA_DIR / "true_negative_labeled_comments_4.json")
true_negative_labeled_comments_4.index = true_negative_labeled_comments_4.index.astype(int)
print(true_negative_labeled_comments_4["label"].value_counts())
true_negative_labeled_comments_4.head()

label
1    134
2    126
0     39
Name: count, dtype: int64


,label
28984,1
139543,1
18746,2
118604,2
90156,0


In [18]:
# Select only the result of negative-manually labeled comments
selected_true_negative_labeled_comments_4 = (
    true_negative_labeled_comments_4[true_negative_labeled_comments_4["label"] == 0]
)

## Negatives - 5

In [19]:
# Get 300 negative-labeled comments only 
sample_df_negative_sentiment_result_5 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "negative"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=5)
)

print(sample_df_negative_sentiment_result_5.shape)
sample_df_negative_sentiment_result_5.head()

(300, 4)


,video_id,comment_id,text,language
106256,OiC1dG6qxyI,Ugy382OPn1CFIimDbGp4AaABAg,this is so funny how han did not win at alle,eng_Latn
192448,XAq2Esnb-ks,UgzSSZW2cF2uO1I14G54AaABAg,"""I.N. is rough on us, but on Felix he is nicer...",eng_Latn
12275,GE-v_p-EXVs,UgwBKUsTa-9ifTOa7R54AaABAg,He really scared me❤😂,eng_Latn
30881,x9TZNpRCeqE,Ugwn4l-ASoXooxaRdkR4AaABAg,stray kids talking to themselves about how the...,eng_Latn
71005,zvDhHMT02Bs,UgxgfdAWf4hLgTXfGMB4AaABAg,THE WAY CHAN CHASED HIM IN THE END MY HEART😭🩵🩵,yue_Hant


In [20]:
# Get the comments only and save them
negative_labeled_comments_5 = sample_df_negative_sentiment_result_5[["text"]]

negative_labeled_comments_5.to_json(INTERIM_DATA_DIR / "negative_labeled_comments_5.json")

In [20]:
# Get the result of manually labeled comments
true_negative_labeled_comments_5 = pd.read_json(INTERIM_DATA_DIR / "true_negative_labeled_comments_5.json")
true_negative_labeled_comments_5.index = true_negative_labeled_comments_5.index.astype(int)
print(true_negative_labeled_comments_5["label"].value_counts())
true_negative_labeled_comments_5.head()

label
2    231
1     53
0     16
Name: count, dtype: int64


,label
106256,2
192448,2
12275,2
30881,2
71005,2


In [22]:
# Select only the result of negative-manually labeled comments
selected_true_negative_labeled_comments_5 = (
    true_negative_labeled_comments_5[true_negative_labeled_comments_5["label"] == 0]
)

## Negatives - 6

In [27]:
# Get 300 negative-labeled comments only 
sample_df_negative_sentiment_result_6 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "negative"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=6)
)

print(sample_df_negative_sentiment_result_6.shape)
sample_df_negative_sentiment_result_6.head()

(300, 4)


,video_id,comment_id,text,language
159461,EHTbxyv7OHA,Ugz6_8ONEDdtyzcsW814AaABAg,"Seungmin's Description ""He looks the most norm...",eng_Latn
85831,7Rd7Y8mWtPY,UgxpqRs1R-wxKigl8sN4AaABAg,I i lose my joobe 😂,eng_Latn
123698,eSRLZfj3X80,UgyHokg_XbK-tJi28mt4AaABAg,NOT PUTTING THIS ON STREAMING PLATFORMS IS A C...,yue_Hant
166732,x9TZNpRCeqE,Ugzcl60nDy8M4XKHdlF4AaABAg,i'm not even kidding when i say changbin needs...,eng_Latn
160947,1EP5hxH6O7M,Ugz7W76Sk9ln7Hmr22F4AaABAg,11:49 용복아 목 상할까 걱정된다ㅠㅠ쪼끔만 살살혀💕,kor_Hang


In [28]:
# Get the comments only and save them
negative_labeled_comments_6 = sample_df_negative_sentiment_result_6[["text"]]

negative_labeled_comments_6.to_json(INTERIM_DATA_DIR / "negative_labeled_comments_6.json")

In [21]:
# Get the result of manually labeled comments
true_negative_labeled_comments_6 = pd.read_json(INTERIM_DATA_DIR / "true_negative_labeled_comments_6.json")
true_negative_labeled_comments_6.index = true_negative_labeled_comments_6.index.astype(int)
print(true_negative_labeled_comments_6["label"].value_counts())
true_negative_labeled_comments_6.head()

label
2    155
1    114
0     31
Name: count, dtype: int64


,label
159461,2
85831,1
123698,2
166732,1
160947,2


In [ ]:
# Select only the result of negative-manually labeled comments
selected_true_negative_labeled_comments_6 = (
    true_negative_labeled_comments_6[true_negative_labeled_comments_6["label"] == 0]
)

## Negatives - 7

In [ ]:
# Get 300 negative-labeled comments only 
sample_df_negative_sentiment_result_7 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "negative"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=7)
)

print(sample_df_negative_sentiment_result_7.shape)
sample_df_negative_sentiment_result_7.head()

(300, 4)


,video_id,comment_id,text,language
137024,u_msRanAhJ8,UgyPYAlmr_FrXGsLJc54AaABAg,JILIXIM 4:10 😭,kor_Hang
9433,w9eOsEFpM6A,Ugw9NNzzdJCHtN25jq54AaABAg,This cool Chan's song is sorely missed on SPO...,eng_Latn
202403,b-ztNXTxxLA,Ugzz006bGrZL_EymO0J4AaABAg,This helium game is just Hyunjin and Han laugh...,eng_Latn
153724,-MmQzchmtmQ,Ugz-sc9cjo7_x23gKft4AaABAg,너무 웃겨서 웃으면 배가 아프다,kor_Hang
96111,tBJLZWvMMYU,UgxWadYkFEfTI39DwvZ4AaABAg,The bad thing is that I'm broke and won't get ...,eng_Latn


In [32]:
# Get the comments only and save them
negative_labeled_comments_7 = sample_df_negative_sentiment_result_7[["text"]]

negative_labeled_comments_7.to_json(INTERIM_DATA_DIR / "negative_labeled_comments_7.json")

In [22]:
# Get the result of manually labeled comments
true_negative_labeled_comments_7 = pd.read_json(INTERIM_DATA_DIR / "true_negative_labeled_comments_7.json")
true_negative_labeled_comments_7.index = true_negative_labeled_comments_7.index.astype(int)
print(true_negative_labeled_comments_7["label"].value_counts())
true_negative_labeled_comments_7.head()

label
2    194
1     63
0     43
Name: count, dtype: int64


,label
137024,2
9433,2
202403,2
153724,2
96111,0


In [ ]:
# Select only the result of negative-manually labeled comments
selected_true_negative_labeled_comments_7 = (
    true_negative_labeled_comments_7[true_negative_labeled_comments_7["label"] == 0]
)

## Negatives - 8

In [3]:
# Get 300 negative-labeled comments only 
sample_df_negative_sentiment_result_8 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "negative"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=8)
)

print(sample_df_negative_sentiment_result_8.shape)
sample_df_negative_sentiment_result_8.head()

(300, 4)


,video_id,comment_id,text,language
21425,IEN8KgAIVVg,UgwhbxiCuVAGGZWdzdt4AaABAg,7:59 i still come back to this video just to h...,eng_Latn
5928,ILH2lECVZ-8,Ugw5eH5hFnL57AsPYk94AaABAg,용복아 ㅜㅜ 이게 무슨일이야ㅠㅠ 제발 푹~ 쉬어 다 나아져도 몇주 더 쉬고 와 제발...,kor_Hang
39958,Ax76YDZ3qns,UgwStcpjipQ22vPQLYB4AaABAg,"""Know"" was like nah I'm not doing this anymore 😭🎀",eng_Latn
44691,UAanagaR_jY,UgwVSBs4ZQuKYCicsPV4AaABAg,Hyunjin unbuttoned the shirt and we all collec...,eng_Latn
41618,4Xv5HJo5AFw,UgwTuXshDUZx15qrkfZ4AaABAg,I'm completely speechless no wonder they are k...,eng_Latn


In [4]:
# Get the comments only and save them
negative_labeled_comments_8 = sample_df_negative_sentiment_result_8[["text"]]

negative_labeled_comments_8.to_json(INTERIM_DATA_DIR / "negative_labeled_comments_8.json")

In [23]:
# Get the result of manually labeled comments
true_negative_labeled_comments_8 = pd.read_json(INTERIM_DATA_DIR / "true_negative_labeled_comments_8.json")
true_negative_labeled_comments_8.index = true_negative_labeled_comments_8.index.astype(int)
print(true_negative_labeled_comments_8["label"].value_counts())
true_negative_labeled_comments_8.head()

label
2    189
1     80
0     30
Name: count, dtype: int64


,label
21425,2
5928,2
39958,1
44691,2
41618,2


In [6]:
# Select only the result of negative-manually labeled comments
selected_true_negative_labeled_comments_8 = (
    true_negative_labeled_comments_8[true_negative_labeled_comments_8["label"] == 0]
)

## Negatives - 9

In [7]:
# Get 300 negative-labeled comments only 
sample_df_negative_sentiment_result_9 = (
    df_sentiment_result[df_sentiment_result["sentiment_label"] == "negative"]
    .drop("sentiment_label", axis=1).sample(n=300, random_state=9)
)

print(sample_df_negative_sentiment_result_9.shape)
sample_df_negative_sentiment_result_9.head()

(300, 4)


,video_id,comment_id,text,language
49483,ECC-hru050A,UgwYt8Ow6t1NGVdOoul4AaABAg,WHAT DO YOU EVEN KNOW 😭😭😭😭,yue_Hant
202649,v_1wQ6ygoYk,UgzzAc84Ix0KvNj-k5Z4AaABAg,Yet another sign from the universe telling me ...,eng_Latn
113882,LPS5GoBgQq8,UgyBIoOJL5j8GPHk0WR4AaABAg,0:27 \nWELL I AM ✨ UNCOMFORTABLE ✨,yue_Hant
181144,VRBiibeXJEk,UgzLoyfxsuoN_x2nb7l4AaABAg,Everyone is talking about why everyone is talk...,eng_Latn
151825,rGuWDQglr1I,Ugyza6w6Z0pqdmhKW_h4AaABAg,Tadinho do felix a galinha não foi nele,por_Latn


In [8]:
# Get the comments only and save them
negative_labeled_comments_9 = sample_df_negative_sentiment_result_9[["text"]]

negative_labeled_comments_9.to_json(INTERIM_DATA_DIR / "negative_labeled_comments_9.json")

In [24]:
# Get the result of manually labeled comments
true_negative_labeled_comments_9 = pd.read_json(INTERIM_DATA_DIR / "true_negative_labeled_comments_9.json")
true_negative_labeled_comments_9.index = true_negative_labeled_comments_9.index.astype(int)
print(true_negative_labeled_comments_9["label"].value_counts())
true_negative_labeled_comments_9.head()

label
1    163
2    125
0     11
Name: count, dtype: int64


,label
49483,1
202649,2
113882,1
181144,2
151825,1


In [10]:
# Select only the result of negative-manually labeled comments
selected_true_negative_labeled_comments_9 = (
    true_negative_labeled_comments_9[true_negative_labeled_comments_9["label"] == 0]
)

## Combining All Samples

In [25]:
combined_true_comments = pd.concat([
    true_positive_labeled_comments_1,
    true_positive_labeled_comments_2,
    true_neutral_labeled_comments_1,
    true_neutral_labeled_comments_2,
    true_neutral_labeled_comments_3,
    true_negative_labeled_comments_1,
    true_negative_labeled_comments_2,
    true_negative_labeled_comments_3,
    true_negative_labeled_comments_4,
    true_negative_labeled_comments_5,
    true_negative_labeled_comments_6,
    true_negative_labeled_comments_7,
    true_negative_labeled_comments_8,
    true_negative_labeled_comments_9
])

combined_true_comments = (
    combined_true_comments.reset_index()
    .drop_duplicates().set_index("index")
)

combined_true_comments.index.name = None

print(combined_true_comments.shape)
print(combined_true_comments["label"].value_counts())
combined_true_comments.head()

(4104, 1)
label
2    2506
1    1286
0     312
Name: count, dtype: int64


,label
50586,2
32845,2
165842,2
113926,2
131793,2


In [26]:
positive_combined_true_comments = combined_true_comments[combined_true_comments["label"] == 2].sample(n= 450, random_state=25)
neutral_combined_true_comments = combined_true_comments[combined_true_comments["label"] == 1].sample(n= 400, random_state=26)
negative_combined_true_comments = combined_true_comments[combined_true_comments["label"] == 0]

selected_true_comments = pd.concat([
    positive_combined_true_comments,
    neutral_combined_true_comments,
    negative_combined_true_comments
])

selected_true_comments = selected_true_comments.sample(frac=1, random_state=50)

print(selected_true_comments.shape)
print(selected_true_comments["label"].value_counts())
selected_true_comments.head()

(1162, 1)
label
2    450
1    400
0    312
Name: count, dtype: int64


,label
64192,2
147349,1
40354,0
51196,0
36313,1


## Joining the labels with the text data

In [27]:
df_true_sentiment_samples = df_sentiment_result[["text"]].join(selected_true_comments, how="right")

print(df_true_sentiment_samples.shape)
print(df_true_sentiment_samples["label"].value_counts())
df_true_sentiment_samples.head()

(1162, 2)
label
2    450
1    400
0    312
Name: count, dtype: int64


,text,label
64192,Chan all worried about making a mistake! 😢 Lit...,2
147349,"""They keep forgetting HAN is talkative""\nHAN j...",1
40354,I missed it🥲😔,0
51196,스테이6기 멤버쉽금액이 …아쉽네요. \n어린소녀팬들은 금액이 너무 부담될듯요. 그...,0
36313,them at green water pot \nhyunjin and bangchan...,1


In [28]:
df_true_sentiment_samples.to_parquet(INTERIM_DATA_DIR / "df_true_sentiment_samples.parquet")

In [30]:
df_true_sentiment_samples[df_true_sentiment_samples["label"] == 0].sample(5)

,text,label
83036,it took Felix only 20 sec more than Han for th...,0
192163,Dear Woojin\nIf you accidentally come across t...,0
40632,Has anyone noticed Felix sitting in an upright...,0
161565,1:15 ALREADY OVER!?!?!??!,0
33780,I hope the contract does not end in 2025. I am...,0


# Evaluation 2 - Fine-Tuned Model

In [1]:
import pandas as pd

from src.config import INTERIM_DATA_DIR

In [4]:
df_sentiment_result = pd.read_parquet(INTERIM_DATA_DIR / "df_sentiment_result.parquet")

df_sentiment_result.sample(10)

,video_id,comment_id,text,language,sentiment_label
7992,uF9WWx0e2t0,Ugw7cbDskyhWQNdny4Z4AaABAg,"너무 귀여워, 아아, 심쿵사할 것 같아, 현진아, 넌 최고야!",kor_Hang,positive
142052,sNG7pALOnpQ,Ugyn6m9e6NdxHTcE5kR4AaABAg,can we talk about his body control holy crap h...,eng_Latn,positive
155963,ADOBeE-Qs5g,UgyvA1SjWGeURShedDl4AaABAg,THE INTERACTION BETWEEN YEJI AND HYUNJIN IS PE...,yue_Hant,positive
59585,9ZXTuvbX8DM,Ugx3lfm2KnFJPiYcpc14AaABAg,I can't put into words just how beautiful and ...,eng_Latn,positive
124532,yL2Dt8wjgJQ,UgycuCNgHMtFZ2B268N4AaABAg,one of the reasons why I hate school: MAKING M...,eng_Latn,negative
218593,ZGoCy3nWg8E,UgzzUwQ-S9NK0jB9Vk14AaABAg,정말 대단한 그룹이고 모든 것에 많은 노력을 기울입니다.그들의 목소리는 나에게 평화...,kor_Hang,positive
140065,c6yyoWhyiWw,UgyLYHQFlMYcTPAECIV4AaABAg,They didn't play around when they said that ST...,eng_Latn,positive
110688,L8J0fqiYnJU,Ugy_bg_dwVR9iHPCRuV4AaABAg,One of the best decision I made is stanning ST...,eng_Latn,positive
29493,IR0KSVV-QGE,Ugwlb2L67a7FMsZpQuF4AaABAg,"""Aah-""\n ~Han, 2025\n\nSuch inspiring words",eng_Latn,positive
24548,T70W-TSIkTQ,UgwIdjSczgjnTCmtFT14AaABAg,찬의 미소는 세상에서 가장 소중한 보물이다.😊😊😊😊😊,kor_Hang,positive


In [16]:
sample_df_sentiment_result = df_sentiment_result[["text"]].sample(n=384, random_state=90)

In [ ]:
sample_df_sentiment_result.to_excel(INTERIM_DATA_DIR / "sample_df_sentiment_result.json")

In [ ]:
true_sample_df_sentiment_result = pd.read_excel(INTERIM_DATA_DIR / "true_sample_df_sentiment_result.json")

In [13]:
true_sample_df_sentiment_result.head()

,true_sentiment_label
8635,positive
62648,positive
73301,negative
126014,positive
136849,positive


In [32]:
labeled_df = sample_df_sentiment_result.join(true_sample_df_sentiment_result).join(df_sentiment_result[["sentiment_label"]])

In [33]:
# Identify the misclassified rows
misclassified = labeled_df[labeled_df["sentiment_label"] != labeled_df["true_sentiment_label"]]

# Calculate the error rate
error_rate = len(misclassified) / len(labeled_df)
print(f"Estimated Error Rate: {error_rate * 100:.2f}%")

Estimated Error Rate: 12.24%


In [34]:
labeled_df[labeled_df["sentiment_label"] != labeled_df["true_sentiment_label"]]

,text,true_sentiment_label,sentiment_label
136849,"I was gonna go to sleep, guess that’s out of t...",positive,neutral
163266,Understanding❌Vibing✅,positive,neutral
107617,minho’s so calm when they’re all literally scr...,neutral,positive
122660,"13:57 ""a relatively normal dog and chick"" then...",neutral,negative
10595,It's funny the face of han when Changbin slapp...,positive,neutral
85484,"*""Maybe I should move to a different planet""*\...",positive,neutral
145110,HIP HIP HOORAY WE ALL SHOUT IN UNISON,positive,neutral
12063,Nah.. this is not dangerous.. this is emotional,neutral,positive
90258,"ATTENTION STAYS!!!!\n\nas you know, bangchan h...",neutral,negative
110209,Rare aesthetic of them doing a masculine cover,neutral,positive


An 87.76% accuracy rate is acceptable. Therefore, the model is performing better and can be used for further analysis. The model is now able to classify comments better into positive, neutral, and negative categories with a reasonable level of confidence.